# Data Cleaning and Quality in SQL

## Quick notes
### What this covers
- Profiling a table: row counts, NULLs, distinct values, min/max, lengths and shapes.
- Finding problems: exact and near duplicates, orphan rows, invalid values, bad dates, outliers.
- Fixing them: trim, case, date and phone standardising with `UPDATE`, merging duplicates safely.
- Doing it safely: transactions, before/after counts, idempotent scripts, audit log, batched backfills.
- Reconciling two tables, the staging-table pattern, and a data-quality scorecard query.
### Key points
- Profile first, fix second. Count the problems before you touch a single row.
- Write each rule twice: once as a `SELECT` that finds the bad rows, once as an `UPDATE` that fixes them. Run the `SELECT` first.
- A good cleaning script is idempotent: running it twice gives the same result as running it once (the second run changes 0 rows).
- Normalise to a key (`lower(trim(email))`) to catch near-duplicates that exact matching misses.
- Fix what you can prove is safe. Flag the rest for a human. Never guess silently.
### Gotchas (the part worth revising twice)
- `COUNT(col)` skips NULLs, `COUNT(*)` does not. The gap between them is your NULL count.
- `NULL` never equals `NULL`. Use `IS NOT` for "different, NULL-safe" comparisons.
- `NOT IN (subquery)` returns nothing if the subquery contains a NULL. Use `NOT EXISTS` or `LEFT JOIN ... IS NULL` for orphan checks.
- `PARTITION BY nullable_col` puts all NULLs in one group, so unrelated rows look like duplicates.
- Delete duplicates only after re-pointing child rows (orders) to the survivor, or you create orphans.
- A text column happily stores `'2024-02-30'`. SQLite's `date()` does not return NULL for it, it rolls it over to `2024-03-01`. So test with `date(x) IS NOT x`, which catches both rolled-over and unparseable values.
- Ambiguous formats (`05/01/2024`) need a decision from the data owner (day first or month first), not a guess.
- `changes()` reports rows touched by the last statement. Read it straight after the `UPDATE`.
### When it matters
- Importing CSVs, merging systems, migrating databases, before building reports, and any time numbers "look wrong".
### One-line recap example
```sql
UPDATE customers SET email = lower(trim(email)) WHERE email IS NOT lower(trim(email));
```

## 01. Seed data: a deliberately messy table
- `customers` is a typical import: stray spaces, mixed case, five phone formats, two date formats, a bad email, an impossible date, impossible ages, an exact duplicate and a near-duplicate.
- `orders` has no declared foreign key (common in imports), so orphans can sneak in.
- For the basics of creating tables see `05-DDL-And-Integrity`.

In [1]:
# setup: an in-memory SQLite database and a %%sql cell magic that prints result tables
import sqlite3
from IPython.core.magic import register_cell_magic

con = sqlite3.connect(":memory:", isolation_level=None)   # autocommit, so BEGIN/COMMIT work as written
con.execute("PRAGMA foreign_keys = ON")

def _table(cols, rows):
    cells = [[("NULL" if v is None else str(v)) for v in r] for r in rows]
    widths = [max([len(c)] + [len(r[i]) for r in cells]) for i, c in enumerate(cols)]
    line = lambda vals: " | ".join(v.ljust(w) for v, w in zip(vals, widths))
    print(line(cols)); print("-+-".join("-" * w for w in widths))
    for r in cells: print(line(r))
    print(f"({len(rows)} row{'s' if len(rows) != 1 else ''})\n")

def _statements(text):
    """Split a cell into statements (handles several per line, triggers, quoted semicolons)."""
    out, start = [], 0
    for i, ch in enumerate(text):
        if ch == ";" and sqlite3.complete_statement(text[start:i + 1]):
            out.append(text[start:i + 1].strip()); start = i + 1
    if text[start:].strip():
        out.append(text[start:].strip())
    return out

@register_cell_magic
def sql(line, cell):
    """Run every statement in the cell; print rows for statements that return them."""
    expect_error = "expect_error" in line
    for stmt in _statements(cell):
        try:
            cur = con.execute(stmt)
            if cur.description:
                _table([d[0] for d in cur.description], cur.fetchall())
        except sqlite3.Error as exc:
            if not expect_error:
                raise
            print(f"ERROR: {exc}\n")

In [2]:
%%sql
CREATE TABLE customers (
    id INTEGER PRIMARY KEY, name TEXT, email TEXT, phone TEXT,
    country TEXT, signup TEXT, age INTEGER, spend REAL
);
INSERT INTO customers VALUES
 (1,'Asha Rao ','ASHA@Example.com','(555) 010-2001','india','2024-01-05',31,120.5),
 (2,'asha rao','asha@example.com','555.010.2001','IN','05/01/2024',31,80),
 (3,'Ben Lee','ben@example.com','555-010-3002','US','2024-02-11',45,200),
 (4,'Cara Diaz','cara@example','5550104003','us','2024-02-30',29,50),
 (5,'Dev Patel',NULL,'555 010 5004','IN','2024-03-01',-4,75),
 (6,'Eli Cho','eli@example.com','','KR','2024-03-15',212,9999),
 (7,'Fay Wong','fay@example.com','555-010-7006','  ','2024-04-02',38,60),
 (8,'Gus Hill','gus@example.com','555-010-8007','US','2024-04-09',52,95),
 (9,'Gus Hill','gus@example.com','555-010-8007','US','2024-04-09',52,95);
CREATE TABLE orders (id INTEGER PRIMARY KEY, customer_id INTEGER, amount REAL);
INSERT INTO orders VALUES (1,1,50),(2,2,30),(3,3,200),(4,99,15),(5,NULL,20),(6,9,10);
SELECT * FROM customers ORDER BY id;

id | name      | email            | phone          | country | signup     | age | spend 
---+-----------+------------------+----------------+---------+------------+-----+-------
1  | Asha Rao  | ASHA@Example.com | (555) 010-2001 | india   | 2024-01-05 | 31  | 120.5 
2  | asha rao  | asha@example.com | 555.010.2001   | IN      | 05/01/2024 | 31  | 80.0  
3  | Ben Lee   | ben@example.com  | 555-010-3002   | US      | 2024-02-11 | 45  | 200.0 
4  | Cara Diaz | cara@example     | 5550104003     | us      | 2024-02-30 | 29  | 50.0  
5  | Dev Patel | NULL             | 555 010 5004   | IN      | 2024-03-01 | -4  | 75.0  
6  | Eli Cho   | eli@example.com  |                | KR      | 2024-03-15 | 212 | 9999.0
7  | Fay Wong  | fay@example.com  | 555-010-7006   |         | 2024-04-02 | 38  | 60.0  
8  | Gus Hill  | gus@example.com  | 555-010-8007   | US      | 2024-04-09 | 52  | 95.0  
9  | Gus Hill  | gus@example.com  | 555-010-8007   | US      | 2024-04-09 | 52  | 95.0  
(9 rows)



## 02. Profile: counts, NULLs, distinct, min and max
- Start every cleaning job with one profiling query per table.
- `COUNT(*) - COUNT(col)` = NULL count. `COUNT(DISTINCT col)` = how many different values.
- Empty strings and blanks are not NULL, so count them separately.

In [3]:
%%sql
SELECT COUNT(*)                          AS total_rows,
       COUNT(email)                      AS email_filled,
       COUNT(*) - COUNT(email)           AS email_null,
       SUM(email = '')                   AS email_empty,
       COUNT(DISTINCT email)             AS email_distinct,
       COUNT(DISTINCT lower(trim(email))) AS email_distinct_normalised,
       SUM(phone = '')                   AS phone_empty,
       SUM(trim(country) = '')           AS country_blank,
       MIN(age) AS age_min, MAX(age) AS age_max,
       MIN(signup) AS signup_min, MAX(signup) AS signup_max
FROM customers;

total_rows | email_filled | email_null | email_empty | email_distinct | email_distinct_normalised | phone_empty | country_blank | age_min | age_max | signup_min | signup_max
-----------+--------------+------------+-------------+----------------+---------------------------+-------------+---------------+---------+---------+------------+-----------
9          | 8            | 1          | 0           | 7              | 6                         | 1           | 1             | -4      | 212     | 05/01/2024 | 2024-04-09
(1 row)



## 03. Profile: lengths and shapes with GLOB
- Length checks catch padding and truncation: `length(name) <> length(trim(name))` means stray spaces.
- A "shape" tells you the format: group by a pattern and count. `GLOB` is case-sensitive and supports `[0-9]`.
- Here we classify every phone and signup value into a shape.

In [4]:
%%sql
SELECT CASE
         WHEN phone = ''                                         THEN 'empty'
         WHEN phone GLOB '[0-9][0-9][0-9]-[0-9][0-9][0-9]-[0-9][0-9][0-9][0-9]' THEN 'NNN-NNN-NNNN'
         WHEN phone GLOB '[0-9][0-9][0-9].[0-9][0-9][0-9].[0-9][0-9][0-9][0-9]' THEN 'NNN.NNN.NNNN'
         WHEN phone GLOB '[0-9][0-9][0-9] [0-9][0-9][0-9] [0-9][0-9][0-9][0-9]' THEN 'NNN NNN NNNN'
         WHEN phone GLOB '[0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9]'   THEN 'NNNNNNNNNN'
         ELSE 'other'
       END AS phone_shape, COUNT(*) AS n
FROM customers GROUP BY phone_shape ORDER BY n DESC, phone_shape;
SELECT CASE WHEN signup GLOB '[0-9][0-9][0-9][0-9]-[0-9][0-9]-[0-9][0-9]' THEN 'ISO yyyy-mm-dd'
            WHEN signup GLOB '[0-9][0-9]/[0-9][0-9]/[0-9][0-9][0-9][0-9]' THEN 'dd/mm/yyyy'
            ELSE 'other' END AS date_shape, COUNT(*) AS n
FROM customers GROUP BY date_shape ORDER BY date_shape;
SELECT id, '[' || name || ']' AS name_in_brackets, length(name) - length(trim(name)) AS extra_spaces
FROM customers WHERE name <> trim(name) ORDER BY id;

phone_shape  | n
-------------+--
NNN-NNN-NNNN | 4
NNN NNN NNNN | 1
NNN.NNN.NNNN | 1
NNNNNNNNNN   | 1
empty        | 1
other        | 1
(6 rows)

date_shape     | n
---------------+--
ISO yyyy-mm-dd | 8
dd/mm/yyyy     | 1
(2 rows)

id | name_in_brackets | extra_spaces
---+------------------+-------------
1  | [Asha Rao ]      | 1           
(1 row)



## 04. Profile: value frequencies
- A frequency table of a low-cardinality column shows spelling variants at a glance.
- `india` and `IN` both mean India, and `us` and `US` are the same country. Group by the raw value to see the mess, by `upper(trim(...))` to see the true counts.
- Even after `upper(trim(...))`, `INDIA` and `IN` are still two values. A mapping table (section 12) fixes that.

In [5]:
%%sql
SELECT '[' || country || ']' AS raw_country, COUNT(*) AS n FROM customers GROUP BY country ORDER BY raw_country;
SELECT upper(trim(country)) AS normalised, COUNT(*) AS n FROM customers GROUP BY 1 ORDER BY 1;

raw_country | n
------------+--
[  ]        | 1
[IN]        | 2
[KR]        | 1
[US]        | 3
[india]     | 1
[us]        | 1
(6 rows)

normalised | n
-----------+--
           | 1
IN         | 2
INDIA      | 1
KR         | 1
US         | 4
(5 rows)



## 05. Exact duplicates
- Group by every column that defines "the same record" (excluding the surrogate `id`) and keep groups with more than one row.
- `GROUP_CONCAT(id)` lists which ids are involved.
- Only rows 8 and 9 match exactly. Rows 1 and 2 are the same person, but the text differs, so this query misses them.

In [6]:
%%sql
SELECT name, email, phone, country, signup, COUNT(*) AS copies, GROUP_CONCAT(id) AS ids
FROM customers
GROUP BY name, email, phone, country, signup
HAVING COUNT(*) > 1
ORDER BY ids;

name     | email           | phone        | country | signup     | copies | ids
---------+-----------------+--------------+---------+------------+--------+----
Gus Hill | gus@example.com | 555-010-8007 | US      | 2024-04-09 | 2      | 8,9
(1 row)



## 06. Near-duplicates with a normalised key
- Build a key that removes noise: `lower(trim(email))`. Rows sharing the key are the same entity.
- Another key for people without an email: normalised name plus digits-only phone.
- `ROW_NUMBER` inside each key group marks the first row to keep and the rest as extras (see `03-Window-Functions/01-Window-Functions`).
- Gotcha: partitioning by a NULL email lumps all NULL-email rows together. Fall back to something unique for them.

In [7]:
%%sql
WITH keyed AS (
    SELECT id, name, email,
           COALESCE(lower(trim(email)), 'no-email-' || id) AS dup_key
    FROM customers
)
SELECT id, name, dup_key,
       ROW_NUMBER() OVER (PARTITION BY dup_key ORDER BY id) AS rn,
       COUNT(*)     OVER (PARTITION BY dup_key)             AS group_size
FROM keyed
ORDER BY dup_key, id;

id | name      | dup_key          | rn | group_size
---+-----------+------------------+----+-----------
1  | Asha Rao  | asha@example.com | 1  | 2         
2  | asha rao  | asha@example.com | 2  | 2         
3  | Ben Lee   | ben@example.com  | 1  | 1         
4  | Cara Diaz | cara@example     | 1  | 1         
6  | Eli Cho   | eli@example.com  | 1  | 1         
7  | Fay Wong  | fay@example.com  | 1  | 1         
8  | Gus Hill  | gus@example.com  | 1  | 2         
9  | Gus Hill  | gus@example.com  | 2  | 2         
5  | Dev Patel | no-email-5       | 1  | 1         
(9 rows)



## 07. Orphan rows and missing foreign keys
- An orphan is a child row whose parent does not exist. A missing FK is a NULL where a parent is required.
- Find orphans with an anti-join: `LEFT JOIN parent ... WHERE parent.id IS NULL` (or `NOT EXISTS`).
- `NOT IN` is dangerous when the list contains a NULL: it returns no rows. It happens to work in the first `NOT IN` query (the next code cell) only because `customers.id` is never NULL. See `02-Querying/01-Joins` for anti-joins.

In [8]:
%%sql
-- Orders pointing at a customer that does not exist (the NULL one is handled separately)
SELECT o.id AS order_id, o.customer_id
FROM orders o LEFT JOIN customers c ON c.id = o.customer_id
WHERE o.customer_id IS NOT NULL AND c.id IS NULL
ORDER BY o.id;
-- Orders with no customer at all
SELECT id AS order_id FROM orders WHERE customer_id IS NULL ORDER BY id;
-- Customers who never ordered (parent without children)
SELECT c.id, c.name FROM customers c
WHERE NOT EXISTS (SELECT 1 FROM orders o WHERE o.customer_id = c.id) ORDER BY c.id;

order_id | customer_id
---------+------------
4        | 99         
(1 row)

order_id
--------
5       
(1 row)

id | name     
---+----------
4  | Cara Diaz
5  | Dev Patel
6  | Eli Cho  
7  | Fay Wong 
8  | Gus Hill 
(5 rows)



In [9]:
%%sql
-- Works here, because customers.id has no NULLs
SELECT id FROM orders WHERE customer_id NOT IN (SELECT id FROM customers) ORDER BY id;
-- Bad: the list now contains a NULL (order 5), so NOT IN returns 0 rows even though customers 4 to 8 never ordered
SELECT id FROM customers WHERE id NOT IN (SELECT customer_id FROM orders) ORDER BY id;

id
--
4 
(1 row)

id
--
(0 rows)



## 08. Invalid values: CHECK-like validation queries
- Write each business rule as a query that returns the violating rows. This is what a `CHECK` constraint would enforce, but for data already in the table.
- Wrap NULL-prone tests in `COALESCE(..., 0)` so "unknown" counts as a failure instead of silently passing.
- Once the data is clean, add the real `CHECK` (see `05-DDL-And-Integrity`).

In [10]:
%%sql
SELECT id, 'age out of range' AS problem, age AS value FROM customers WHERE age NOT BETWEEN 0 AND 120
UNION ALL
SELECT id, 'email missing',  email FROM customers WHERE email IS NULL OR email = ''
UNION ALL
SELECT id, 'email malformed', email FROM customers WHERE email NOT LIKE '%_@_%._%'
UNION ALL
SELECT id, 'negative or huge spend', spend FROM customers WHERE spend < 0 OR spend > 1000
ORDER BY id, problem;

id | problem                | value       
---+------------------------+-------------
4  | email malformed        | cara@example
5  | age out of range       | -4          
5  | email missing          | NULL        
6  | age out of range       | 212         
6  | negative or huge spend | 9999.0      
(5 rows)



## 09. Impossible dates
- SQLite stores dates as text, so `'2024-02-30'` is accepted. `date()` returns NULL for text it cannot parse (such as `05/01/2024`) and silently rolls impossible days over (`2024-02-30` becomes `2024-03-01`).
- `date(x) IS NOT x` is true for both cases, so it finds every value that is not a clean, real ISO date.
- There is no `DATEDIFF`: use `julianday(a) - julianday(b)` for days between two dates.

In [11]:
%%sql
SELECT id, signup, date(signup) AS parsed,
       CASE WHEN date(signup) IS NULL THEN 'not ISO text'
            ELSE 'impossible day, rolled over' END AS verdict
FROM customers WHERE date(signup) IS NOT signup ORDER BY id;
SELECT julianday('2024-04-09') - julianday('2024-01-05') AS days_between;

id | signup     | parsed     | verdict                    
---+------------+------------+----------------------------
2  | 05/01/2024 | NULL       | not ISO text               
4  | 2024-02-30 | 2024-03-01 | impossible day, rolled over
(2 rows)

days_between
------------
95.0        
(1 row)



## 10. Outliers with the IQR rule
- IQR rule: a value is an outlier if it is below Q1 - 1.5*IQR or above Q3 + 1.5*IQR, where IQR = Q3 - Q1.
- SQLite has no `PERCENTILE`. Build quartiles with `ROW_NUMBER` over the sorted values and pick the row at position `ceil(p * n)` (the nearest-rank method).
- Integer ceiling trick: `(n * 25 + 99) / 100` is `ceil(n * 0.25)`.
- Here Ben (200) and Eli (9999) are flagged. Eli looks like a typo, Ben may be a genuine big spender.
- Outliers are flagged for review, not deleted. They can be real (a big customer) or typos.
- Note the duplicate (row 9) is still in the data, so it nudges the quartiles. Dedupe first in real work.

In [12]:
%%sql
WITH ranked AS (
    SELECT id, spend,
           ROW_NUMBER() OVER (ORDER BY spend, id) AS rn,
           COUNT(*)     OVER ()                   AS n
    FROM customers
),
q AS (
    SELECT (SELECT spend FROM ranked WHERE rn = (n * 25 + 99) / 100) AS q1,
           (SELECT spend FROM ranked WHERE rn = (n * 75 + 99) / 100) AS q3
    FROM ranked LIMIT 1
)
SELECT q1, q3, q3 - q1 AS iqr,
       q1 - 1.5 * (q3 - q1) AS lower_fence,
       q3 + 1.5 * (q3 - q1) AS upper_fence
FROM q;

q1   | q3    | iqr  | lower_fence | upper_fence
-----+-------+------+-------------+------------
75.0 | 120.5 | 45.5 | 6.75        | 188.75     
(1 row)



In [13]:
%%sql
WITH ranked AS (
    SELECT id, spend, ROW_NUMBER() OVER (ORDER BY spend, id) AS rn, COUNT(*) OVER () AS n FROM customers
),
q AS (
    SELECT (SELECT spend FROM ranked WHERE rn = (n * 25 + 99) / 100) AS q1,
           (SELECT spend FROM ranked WHERE rn = (n * 75 + 99) / 100) AS q3
    FROM ranked LIMIT 1
)
SELECT c.id, c.name, c.spend
FROM customers c, q
WHERE c.spend < q.q1 - 1.5 * (q.q3 - q.q1) OR c.spend > q.q3 + 1.5 * (q.q3 - q.q1)
ORDER BY c.spend;

id | name    | spend 
---+---------+-------
3  | Ben Lee | 200.0 
6  | Eli Cho | 9999.0
(2 rows)



## 11. Data-quality scorecard (before cleaning)
- Put the row-level checks in a view, one 0/1 column per rule, then summarise them in one query: failing rows and pass percentage per rule.
- Use the same view before and after cleaning to prove the improvement. We run it now and again at the end.
- `COALESCE(test, 0)` makes NULL count as a failure.

In [14]:
%%sql
CREATE VIEW v_checks AS
SELECT id,
       COALESCE(name = trim(name) AND name <> '', 0)                                AS name_ok,
       COALESCE(email LIKE '%_@_%._%' AND email = lower(email), 0)                 AS email_ok,
       COALESCE(phone GLOB '[0-9][0-9][0-9]-[0-9][0-9][0-9]-[0-9][0-9][0-9][0-9]', 0) AS phone_ok,
       COALESCE(country GLOB '[A-Z][A-Z]', 0)                                       AS country_ok,
       COALESCE(date(signup) = signup, 0)                                           AS signup_ok,
       COALESCE(age BETWEEN 0 AND 120, 0)                                           AS age_ok
FROM customers;
CREATE VIEW v_scorecard AS
SELECT 'name_ok' AS rule, COUNT(*) - SUM(name_ok) AS failing, COUNT(*) AS total, ROUND(100.0 * SUM(name_ok) / COUNT(*), 1) AS pct_pass FROM v_checks
UNION ALL SELECT 'email_ok',   COUNT(*) - SUM(email_ok),   COUNT(*), ROUND(100.0 * SUM(email_ok)   / COUNT(*), 1) FROM v_checks
UNION ALL SELECT 'phone_ok',   COUNT(*) - SUM(phone_ok),   COUNT(*), ROUND(100.0 * SUM(phone_ok)   / COUNT(*), 1) FROM v_checks
UNION ALL SELECT 'country_ok', COUNT(*) - SUM(country_ok), COUNT(*), ROUND(100.0 * SUM(country_ok) / COUNT(*), 1) FROM v_checks
UNION ALL SELECT 'signup_ok',  COUNT(*) - SUM(signup_ok),  COUNT(*), ROUND(100.0 * SUM(signup_ok)  / COUNT(*), 1) FROM v_checks
UNION ALL SELECT 'age_ok',     COUNT(*) - SUM(age_ok),     COUNT(*), ROUND(100.0 * SUM(age_ok)     / COUNT(*), 1) FROM v_checks;
SELECT * FROM v_scorecard ORDER BY rule;

rule       | failing | total | pct_pass
-----------+---------+-------+---------
age_ok     | 2       | 9     | 77.8    
country_ok | 3       | 9     | 66.7    
email_ok   | 3       | 9     | 66.7    
name_ok    | 1       | 9     | 88.9    
phone_ok   | 5       | 9     | 44.4    
signup_ok  | 2       | 9     | 77.8    
(6 rows)



## 12. Normalise first as a SELECT: the "preview" view
- Put every fix into a view that computes the cleaned values. Then you can preview (`SELECT`), diff against the original, and finally apply (`UPDATE ... FROM`).
- Text: `trim` for spaces, `lower` for emails, `upper` for country codes, `NULLIF(x, '')` turns empty strings into real NULLs.
- Country spellings go through a small mapping table. Unknown values pass through unchanged so a human can see them.
- Dates: only convert `dd/mm/yyyy` to ISO. This assumes day first. Confirm that with the data owner.
- Phones: remove spaces, brackets, dots and dashes, and if exactly 10 digits remain, rewrite as `NNN-NNN-NNNN`. Anything else is kept as is (or NULL if empty) for review.

In [15]:
%%sql
CREATE TABLE country_map (raw TEXT PRIMARY KEY, code TEXT NOT NULL);
INSERT INTO country_map VALUES ('INDIA','IN'),('USA','US');
CREATE VIEW v_norm AS
WITH digits AS (
    SELECT id, replace(replace(replace(replace(replace(phone,' ',''),'(',''),')',''),'.',''),'-','') AS d FROM customers
)
SELECT c.id,
       trim(c.name)                                   AS name,
       NULLIF(lower(trim(c.email)), '')               AS email,
       CASE WHEN length(g.d) = 10 THEN substr(g.d,1,3) || '-' || substr(g.d,4,3) || '-' || substr(g.d,7,4)
            ELSE NULLIF(c.phone, '') END              AS phone,
       CASE WHEN trim(c.country) = '' THEN NULL
            ELSE COALESCE(m.code, upper(trim(c.country))) END AS country,
       CASE WHEN c.signup GLOB '[0-9][0-9]/[0-9][0-9]/[0-9][0-9][0-9][0-9]'
            THEN substr(c.signup,7,4) || '-' || substr(c.signup,4,2) || '-' || substr(c.signup,1,2)
            ELSE c.signup END                         AS signup
FROM customers c
JOIN digits g ON g.id = c.id
LEFT JOIN country_map m ON m.raw = upper(trim(c.country));
-- Preview: only the columns that would change
SELECT c.id, c.name AS old_name, n.name AS new_name, c.email AS old_email, n.email AS new_email,
       c.phone AS old_phone, n.phone AS new_phone
FROM customers c JOIN v_norm n USING (id)
WHERE c.phone IS NOT n.phone OR c.email IS NOT n.email OR c.name IS NOT n.name
ORDER BY c.id;

id | old_name  | new_name  | old_email        | new_email        | old_phone      | new_phone   
---+-----------+-----------+------------------+------------------+----------------+-------------
1  | Asha Rao  | Asha Rao  | ASHA@Example.com | asha@example.com | (555) 010-2001 | 555-010-2001
2  | asha rao  | asha rao  | asha@example.com | asha@example.com | 555.010.2001   | 555-010-2001
4  | Cara Diaz | Cara Diaz | cara@example     | cara@example     | 5550104003     | 555-010-4003
5  | Dev Patel | Dev Patel | NULL             | NULL             | 555 010 5004   | 555-010-5004
6  | Eli Cho   | Eli Cho   | eli@example.com  | eli@example.com  |                | NULL        
(5 rows)



In [16]:
%%sql
SELECT c.id, c.country AS old_country, n.country AS new_country, c.signup AS old_signup, n.signup AS new_signup
FROM customers c JOIN v_norm n USING (id)
WHERE c.country IS NOT n.country OR c.signup IS NOT n.signup
ORDER BY c.id;

id | old_country | new_country | old_signup | new_signup
---+-------------+-------------+------------+-----------
1  | india       | IN          | 2024-01-05 | 2024-01-05
2  | IN          | IN          | 05/01/2024 | 2024-01-05
4  | us          | US          | 2024-02-30 | 2024-02-30
7  |             | NULL        | 2024-04-02 | 2024-04-02
(4 rows)



## 13. Audit trail: record every change
- Before changing data in bulk, make sure you can answer "what was it before?". An audit table plus an `AFTER UPDATE` trigger logs each changed column (see `06-Data-Modification` for triggers).
- `OLD.col IS NOT NEW.col` is the NULL-safe "changed" test.

In [17]:
%%sql
CREATE TABLE audit_log (id INTEGER PRIMARY KEY, row_id INTEGER, col TEXT, old_val TEXT, new_val TEXT);
CREATE TRIGGER trg_customers_audit AFTER UPDATE ON customers
BEGIN
    INSERT INTO audit_log (row_id, col, old_val, new_val) SELECT OLD.id, 'name',    OLD.name,    NEW.name    WHERE OLD.name    IS NOT NEW.name;
    INSERT INTO audit_log (row_id, col, old_val, new_val) SELECT OLD.id, 'email',   OLD.email,   NEW.email   WHERE OLD.email   IS NOT NEW.email;
    INSERT INTO audit_log (row_id, col, old_val, new_val) SELECT OLD.id, 'phone',   OLD.phone,   NEW.phone   WHERE OLD.phone   IS NOT NEW.phone;
    INSERT INTO audit_log (row_id, col, old_val, new_val) SELECT OLD.id, 'country', OLD.country, NEW.country WHERE OLD.country IS NOT NEW.country;
    INSERT INTO audit_log (row_id, col, old_val, new_val) SELECT OLD.id, 'signup',  OLD.signup,  NEW.signup  WHERE OLD.signup  IS NOT NEW.signup;
END;
SELECT COUNT(*) AS audit_rows_so_far FROM audit_log;

audit_rows_so_far
-----------------
0                
(1 row)



## 14. Idempotent cleaning in a transaction, with before/after counts
- Wrap the whole job in `BEGIN ... COMMIT`. If a check fails, `ROLLBACK` and nothing changed (see `06-Data-Modification` for transactions).
- Guard the `UPDATE` with `WHERE col IS NOT new_value`. Rows already clean are skipped, so a second run touches 0 rows. That is idempotence.
- Order matters: normalise first (so near-duplicates become visible), then re-point child rows, then delete duplicates.
- The duplicate map keeps the lowest `id` of each normalised-email group.

In [18]:
%%sql
BEGIN;
SELECT 'before' AS stage, COUNT(*) AS customers, (SELECT SUM(phone_ok) FROM v_checks) AS phones_ok FROM customers;
UPDATE customers SET name = n.name, email = n.email, phone = n.phone, country = n.country, signup = n.signup
FROM v_norm n
WHERE n.id = customers.id
  AND (customers.name    IS NOT n.name    OR customers.email  IS NOT n.email
    OR customers.phone   IS NOT n.phone   OR customers.country IS NOT n.country
    OR customers.signup  IS NOT n.signup);
SELECT changes() AS rows_normalised;
CREATE TEMP TABLE dup_map AS
SELECT id AS dup_id, keep_id FROM (
    SELECT id, FIRST_VALUE(id) OVER (PARTITION BY COALESCE(email, 'no-email-' || id) ORDER BY id) AS keep_id FROM customers
) WHERE id <> keep_id;
SELECT * FROM dup_map ORDER BY dup_id;

stage  | customers | phones_ok
-------+-----------+----------
before | 9         | 4        
(1 row)

rows_normalised
---------------
6              
(1 row)

dup_id | keep_id
-------+--------
2      | 1      
9      | 8      
(2 rows)



In [19]:
%%sql
UPDATE orders SET customer_id = (SELECT keep_id FROM dup_map WHERE dup_id = orders.customer_id)
WHERE customer_id IN (SELECT dup_id FROM dup_map);
SELECT changes() AS orders_repointed;
DELETE FROM customers WHERE id IN (SELECT dup_id FROM dup_map);
SELECT changes() AS duplicates_deleted;
DROP TABLE dup_map;
SELECT 'after' AS stage, COUNT(*) AS customers, (SELECT SUM(phone_ok) FROM v_checks) AS phones_ok FROM customers;
COMMIT;
SELECT id, customer_id, amount FROM orders ORDER BY id;

orders_repointed
----------------
2               
(1 row)

duplicates_deleted
------------------
2                 
(1 row)

stage | customers | phones_ok
------+-----------+----------
after | 7         | 6        
(1 row)

id | customer_id | amount
---+-------------+-------
1  | 1           | 50.0  
2  | 1           | 30.0  
3  | 3           | 200.0 
4  | 99          | 15.0  
5  | NULL        | 20.0  
6  | 8           | 10.0  
(6 rows)



## 15. Prove it is idempotent: run the same job again
- The same statements, second run. Every counter must say 0 and the row count must not move.
- If a re-run changes rows, your rules are not stable (for example a rule that appends text each time).

In [20]:
%%sql
BEGIN;
UPDATE customers SET name = n.name, email = n.email, phone = n.phone, country = n.country, signup = n.signup
FROM v_norm n
WHERE n.id = customers.id
  AND (customers.name    IS NOT n.name    OR customers.email  IS NOT n.email
    OR customers.phone   IS NOT n.phone   OR customers.country IS NOT n.country
    OR customers.signup  IS NOT n.signup);
SELECT changes() AS rows_normalised_second_run;
CREATE TEMP TABLE dup_map AS
SELECT id AS dup_id, keep_id FROM (
    SELECT id, FIRST_VALUE(id) OVER (PARTITION BY COALESCE(email, 'no-email-' || id) ORDER BY id) AS keep_id FROM customers
) WHERE id <> keep_id;
SELECT COUNT(*) AS duplicates_found_second_run FROM dup_map;
DROP TABLE dup_map;
COMMIT;
SELECT COUNT(*) AS customers_now FROM customers;

rows_normalised_second_run
--------------------------
0                         
(1 row)

duplicates_found_second_run
---------------------------
0                          
(1 row)

customers_now
-------------
7            
(1 row)



## 16. Roll back a bad fix
- A rule that is too greedy is the classic cleaning accident. In a transaction you can look at the damage and undo it.
- Here a careless "fix" sets every age above 100 to 100. Row 6 (age 212) is a typo we cannot guess, so the honest fix is NULL plus a flag, not an invented number. We check, then `ROLLBACK`.

In [21]:
%%sql
BEGIN;
UPDATE customers SET age = 100 WHERE age > 100;
SELECT id, age AS age_after_bad_fix FROM customers WHERE id = 6;
ROLLBACK;
SELECT id, age AS age_after_rollback FROM customers WHERE id = 6;

id | age_after_bad_fix
---+------------------
6  | 100              
(1 row)

id | age_after_rollback
---+-------------------
6  | 212               
(1 row)



## 17. What the audit log recorded
- Every changed value is in `audit_log` with old and new. You can summarise it, or undo a column from it.
- The rolled-back age update left no trace. The trigger does not watch `age`, and a rollback would undo any log rows together with the update anyway.

In [22]:
%%sql
SELECT col, COUNT(*) AS changes FROM audit_log GROUP BY col ORDER BY col;
SELECT row_id, col, old_val, new_val FROM audit_log WHERE row_id IN (1, 4) ORDER BY row_id, col;

col     | changes
--------+--------
country | 3      
email   | 1      
name    | 1      
phone   | 5      
signup  | 1      
(5 rows)

row_id | col     | old_val          | new_val         
-------+---------+------------------+-----------------
1      | country | india            | IN              
1      | email   | ASHA@Example.com | asha@example.com
1      | name    | Asha Rao         | Asha Rao        
1      | phone   | (555) 010-2001   | 555-010-2001    
4      | country | us               | US              
4      | phone   | 5550104003       | 555-010-4003    
(6 rows)



## 18. Safe backfill in batches
- Adding a column and filling it for millions of rows in one go can lock the database for a long time. Fill in small batches and repeat until 0 rows change.
- SQLite has no `UPDATE ... LIMIT` by default, so pick ids with `WHERE id IN (SELECT ... ORDER BY id LIMIT k)`.
- Restrict the batch to rows that can actually be filled. Otherwise unmappable rows are picked again forever and the loop never ends.
- Re-running a finished backfill is harmless (idempotent).

In [23]:
%%sql
ALTER TABLE customers ADD COLUMN region TEXT;
CREATE TABLE region_map (country TEXT PRIMARY KEY, region TEXT NOT NULL);
INSERT INTO region_map VALUES ('IN','APAC'),('KR','APAC'),('US','AMER');
UPDATE customers SET region = (SELECT region FROM region_map r WHERE r.country = customers.country)
WHERE region IS NULL AND id IN (
    SELECT c.id FROM customers c JOIN region_map r ON r.country = c.country
    WHERE c.region IS NULL ORDER BY c.id LIMIT 3);
SELECT changes() AS batch_1;

batch_1
-------
3      
(1 row)



In [24]:
%%sql
UPDATE customers SET region = (SELECT region FROM region_map r WHERE r.country = customers.country)
WHERE region IS NULL AND id IN (
    SELECT c.id FROM customers c JOIN region_map r ON r.country = c.country
    WHERE c.region IS NULL ORDER BY c.id LIMIT 3);
SELECT changes() AS batch_2;
UPDATE customers SET region = (SELECT region FROM region_map r WHERE r.country = customers.country)
WHERE region IS NULL AND id IN (
    SELECT c.id FROM customers c JOIN region_map r ON r.country = c.country
    WHERE c.region IS NULL ORDER BY c.id LIMIT 3);
SELECT changes() AS batch_3_nothing_left;
SELECT id, country, region FROM customers ORDER BY id;

batch_2
-------
3      
(1 row)

batch_3_nothing_left
--------------------
0                   
(1 row)

id | country | region
---+---------+-------
1  | IN      | APAC  
3  | US      | AMER  
4  | US      | AMER  
5  | IN      | APAC  
6  | KR      | APAC  
7  | NULL    | NULL  
8  | US      | AMER  
(7 rows)



## 19. Reconcile two tables: the full diff
- Reconciliation = compare two sources of the same facts (books vs bank, old system vs new) and list what is missing on either side or different.
- `FULL OUTER JOIN` (SQLite 3.39+) shows both sides in one query. `IS NOT` compares values NULL-safely.
- Classify each key: only in A, only in B, in both but different, or matching. Summarise with `GROUP BY status`.

In [25]:
%%sql
CREATE TABLE books (id INTEGER PRIMARY KEY, amount REAL);
CREATE TABLE bank  (id INTEGER PRIMARY KEY, amount REAL);
INSERT INTO books VALUES (1,100),(2,250),(3,75),(4,40);
INSERT INTO bank  VALUES (1,100),(2,205),(4,40),(5,60);
CREATE VIEW v_recon AS
SELECT COALESCE(a.id, b.id) AS id, a.amount AS books_amt, b.amount AS bank_amt,
       CASE WHEN b.id IS NULL THEN 'only in books'
            WHEN a.id IS NULL THEN 'only in bank'
            WHEN a.amount IS NOT b.amount THEN 'amount differs'
            ELSE 'match' END AS status
FROM books a FULL OUTER JOIN bank b ON a.id = b.id;
SELECT * FROM v_recon ORDER BY id;
SELECT status, COUNT(*) AS n FROM v_recon GROUP BY status ORDER BY status;

id | books_amt | bank_amt | status        
---+-----------+----------+---------------
1  | 100.0     | 100.0    | match         
2  | 250.0     | 205.0    | amount differs
3  | 75.0      | NULL     | only in books 
4  | 40.0      | 40.0     | match         
5  | NULL      | 60.0     | only in bank  
(5 rows)

status         | n
---------------+--
amount differs | 1
match          | 2
only in bank   | 1
only in books  | 1
(4 rows)



## 20. Staging-table pattern: load raw, clean, then insert
- Never load a messy file straight into the real table. Load it as all-`TEXT` into a staging table (nothing can fail), then clean and validate with SQL, then insert the good rows and park the bad ones in a rejects table with a reason.
- The target table has real constraints (`NOT NULL`, `UNIQUE`, `CHECK`), so anything that sneaks past the validation is still stopped (see `05-DDL-And-Integrity`).
- Rule order in the reason `CASE` decides which reason a row with several problems gets.

In [26]:
%%sql
CREATE TABLE stg_customers (row_no INTEGER PRIMARY KEY, name TEXT, email TEXT, age TEXT, signup TEXT);
INSERT INTO stg_customers VALUES
 (1,'Ann Fox ','ANN@X.COM','31','2024-05-01'),
 (2,'ann fox','ann@x.com','31','01/05/2024'),
 (3,'Bo Ray','bo@x.com','abc','2024-05-02'),
 (4,'Cy Dunn','cy-at-x.com','40','2024-05-03'),
 (5,'Di Eng','di@x.com','28','2024-13-45'),
 (6,'Ed Fry','ed@x.com','55','2024-05-06');
CREATE TABLE customers_v2 (
    id INTEGER PRIMARY KEY, name TEXT NOT NULL, email TEXT NOT NULL UNIQUE CHECK (email LIKE '%_@_%._%'),
    age INTEGER CHECK (age BETWEEN 0 AND 120), signup TEXT CHECK (date(signup) = signup)
);
CREATE TABLE rejects (row_no INTEGER, raw_email TEXT, reason TEXT);
CREATE VIEW stg_final AS
WITH norm AS (
    SELECT row_no, trim(name) AS name, lower(trim(email)) AS email,
           CASE WHEN age GLOB '[0-9]*' AND age NOT GLOB '*[^0-9]*' THEN CAST(age AS INTEGER) END AS age,
           CASE WHEN signup GLOB '[0-9][0-9]/[0-9][0-9]/[0-9][0-9][0-9][0-9]'
                THEN substr(signup,7,4) || '-' || substr(signup,4,2) || '-' || substr(signup,1,2) ELSE signup END AS signup
    FROM stg_customers
), ranked AS (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY email ORDER BY row_no) AS rn FROM norm
)
SELECT row_no, name, email, age, signup,
       CASE WHEN email NOT LIKE '%_@_%._%'            THEN 'bad email'
            WHEN age IS NULL OR age NOT BETWEEN 0 AND 120 THEN 'bad age'
            WHEN date(signup) IS NOT signup           THEN 'bad date'
            WHEN rn > 1                               THEN 'duplicate' END AS reason
FROM ranked;
SELECT * FROM stg_final ORDER BY row_no;

row_no | name    | email       | age  | signup     | reason   
-------+---------+-------------+------+------------+----------
1      | Ann Fox | ann@x.com   | 31   | 2024-05-01 | NULL     
2      | ann fox | ann@x.com   | 31   | 2024-05-01 | duplicate
3      | Bo Ray  | bo@x.com    | NULL | 2024-05-02 | bad age  
4      | Cy Dunn | cy-at-x.com | 40   | 2024-05-03 | bad email
5      | Di Eng  | di@x.com    | 28   | 2024-13-45 | bad date 
6      | Ed Fry  | ed@x.com    | 55   | 2024-05-06 | NULL     
(6 rows)



In [27]:
%%sql
INSERT INTO customers_v2 (name, email, age, signup)
SELECT name, email, age, signup FROM stg_final WHERE reason IS NULL ORDER BY row_no;
INSERT INTO rejects SELECT row_no, email, reason FROM stg_final WHERE reason IS NOT NULL;
SELECT (SELECT COUNT(*) FROM stg_customers) AS staged,
       (SELECT COUNT(*) FROM customers_v2)  AS loaded,
       (SELECT COUNT(*) FROM rejects)       AS rejected;
SELECT * FROM customers_v2 ORDER BY id;
SELECT * FROM rejects ORDER BY row_no;

staged | loaded | rejected
-------+--------+---------
6      | 2      | 4       
(1 row)

id | name    | email     | age | signup    
---+---------+-----------+-----+-----------
1  | Ann Fox | ann@x.com | 31  | 2024-05-01
2  | Ed Fry  | ed@x.com  | 55  | 2024-05-06
(2 rows)

row_no | raw_email   | reason   
-------+-------------+----------
2      | ann@x.com   | duplicate
3      | bo@x.com    | bad age  
4      | cy-at-x.com | bad email
5      | di@x.com    | bad date 
(4 rows)



## 21. Scorecard again (after cleaning)
- Same view, same query as section 11. Compare. Everything we could fix safely now passes.
- What still fails needs a human: the bad email (id 4), the missing email (id 5), the impossible date (id 4), the impossible ages (ids 5 and 6), the missing phone (id 6) and the blank country (id 7). Rule of thumb: automate the provable, flag the rest.

In [28]:
%%sql
SELECT * FROM v_scorecard ORDER BY rule;

rule       | failing | total | pct_pass
-----------+---------+-------+---------
age_ok     | 2       | 7     | 71.4    
country_ok | 1       | 7     | 85.7    
email_ok   | 2       | 7     | 71.4    
name_ok    | 0       | 7     | 100.0   
phone_ok   | 1       | 7     | 85.7    
signup_ok  | 1       | 7     | 85.7    
(6 rows)



In [29]:
%%sql
SELECT c.id, c.name,
       (1 - v.email_ok) + (1 - v.phone_ok) + (1 - v.country_ok) + (1 - v.signup_ok) + (1 - v.age_ok) AS problems
FROM customers c JOIN v_checks v USING (id)
WHERE v.email_ok + v.phone_ok + v.country_ok + v.signup_ok + v.age_ok < 5
ORDER BY problems DESC, c.id;

id | name      | problems
---+-----------+---------
4  | Cara Diaz | 2       
5  | Dev Patel | 2       
6  | Eli Cho   | 2       
7  | Fay Wong  | 1       
(4 rows)

